# 🤟 BISINDO YOLOv11-nano + EigenCAM — Pipeline Lengkap

**Notebook ini menjalankan seluruh pipeline penelitian:**
1. Setup environment & clone repo
2. Mount Google Drive (untuk simpan hasil training)
3. Install dependencies
4. Download dataset BISINDO dari Roboflow
5. Training YOLOv11-nano (40 epoch)
6. Evaluasi model (mAP, Precision, Recall, Speed)
7. Visualisasi EigenCAM XAI

---
⚠️ **Pastikan runtime menggunakan GPU**: `Runtime → Change runtime type → T4 GPU`

## ✅ CEK GPU DULU

In [ ]:
# Cek apakah GPU tersedia
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ GPU tidak terdeteksi! Ganti runtime ke T4 GPU dulu.')

## 📁 STEP 1 — Mount Google Drive

**Kenapa perlu?** Colab free bisa disconnect sewaktu-waktu. Kalau hasil training disimpan di Drive, tidak akan hilang meski sesi putus.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Buat folder project di Drive (kalau belum ada)
import os
DRIVE_PROJECT = '/content/drive/MyDrive/bisindo_project'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print(f'✅ Drive terhubung. Folder project: {DRIVE_PROJECT}')

## 📦 STEP 2 — Clone Repository dari GitHub

In [ ]:
import os

REPO_URL  = 'https://github.com/AlvisChrs/bisindo-yolov11-eigencam.git'
REPO_DIR  = '/content/bisindo-yolov11-eigencam'

if os.path.exists(REPO_DIR):
    # Kalau sudah ada (sesi lanjutan), cukup pull update terbaru
    print('📂 Repo sudah ada, pull update...')
    !git -C {REPO_DIR} pull
else:
    print('📥 Clone repo...')
    !git clone {REPO_URL} {REPO_DIR}

# Pindah ke folder repo
os.chdir(REPO_DIR)
print(f'\n✅ Working directory: {os.getcwd()}')
!ls

## 🔧 STEP 3 — Install Dependencies

In [ ]:
# Install semua library yang diperlukan
# Colab sudah punya torch versi CUDA, jadi kita install sisanya saja
!pip install ultralytics==8.4.123 roboflow==1.4.1 python-dotenv==1.2.3 -q

print('\n✅ Dependencies terinstall!')

# Verifikasi torch + CUDA
import torch
print(f'PyTorch versi : {torch.__version__}')
print(f'CUDA tersedia : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU           : {torch.cuda.get_device_name(0)}')

## 🔑 STEP 4 — Isi Roboflow API Key

**Cara dapat API key:**
1. Buka https://app.roboflow.com
2. Login → klik foto profil → Settings
3. Tab **API Keys** → copy key-nya

In [ ]:
# Isi API key di sini (ganti YOUR_API_KEY_HERE)
ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'  # ← GANTI INI

# Tulis ke file .env supaya script Python bisa baca
with open('.env', 'w') as f:
    f.write(f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')

print('✅ API key tersimpan di .env')

## 📊 STEP 5 — Download Dataset BISINDO dari Roboflow

In [ ]:
# Download dataset (auto-skip kalau sudah ada)
!python -m src.data.download_dataset

# Verifikasi struktur dataset
import os
dataset_dir = 'datasets/bisindo-dataset-1'
if os.path.exists(dataset_dir):
    for split in ['train', 'valid', 'test']:
        path = f'{dataset_dir}/{split}/images'
        if os.path.exists(path):
            n = len(os.listdir(path))
            print(f'  {split:6}: {n} gambar')
    print('\n✅ Dataset siap!')
else:
    print('❌ Dataset belum ada, cek error di atas')

## 🧠 STEP 6 — Training YOLOv11-nano (40 Epoch)

**PENTING untuk Colab free tier:**
- Colab bisa disconnect kapan saja (biasanya 1-4 jam)
- **Kalau putus: cukup jalankan cell ini lagi** — script otomatis lanjut dari checkpoint terakhir!
- Tidak perlu ubah apapun, tidak perlu ingat sudah di epoch berapa

Estimasi waktu di T4 GPU: **~30-60 menit** untuk 40 epoch

In [ ]:
# Training (auto-resume kalau sesi sebelumnya putus)
!python -m src.training.train \
    --config configs/train_config.yaml \
    --project /content/drive/MyDrive/bisindo_project/results \
    --save-period 1

> 💡 **`--save-period 1`** artinya simpan checkpoint tiap epoch ke Drive.
> Kalau Colab putus di epoch 25, Anda tidak kehilangan progress — tinggal jalankan lagi dan lanjut dari epoch 25.
>
> **`--project`** diarahkan ke Google Drive supaya hasil tidak hilang saat sesi Colab berakhir.

## 📈 STEP 7 — Evaluasi Model

Jalankan setelah training selesai. Akan menghitung: **mAP@50, Precision, Recall, Inference Speed**

In [ ]:
# Path ke model terbaik hasil training
BEST_PT = '/content/drive/MyDrive/bisindo_project/results/bisindo_yolo11n/weights/best.pt'

import os
if not os.path.exists(BEST_PT):
    print(f'❌ best.pt tidak ditemukan di: {BEST_PT}')
    print('   Pastikan training sudah selesai!')
else:
    print(f'✅ Model ditemukan: {BEST_PT}')
    
    !python -m src.evaluation.evaluate \
        --weights {BEST_PT} \
        --split test \
        --output /content/drive/MyDrive/bisindo_project/results/evaluation

In [ ]:
# Tampilkan ringkasan metrik
import json

metrics_path = '/content/drive/MyDrive/bisindo_project/results/evaluation/metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        m = json.load(f)

    print('=' * 45)
    print('HASIL EVALUASI — BISINDO YOLOv11-nano')
    print('=' * 45)
    print(f"mAP@50     : {m['map50']*100:.2f}%")
    print(f"mAP@50-95  : {m['map50_95']*100:.2f}%")
    print(f"Precision  : {m['precision']*100:.2f}%")
    print(f"Recall     : {m['recall']*100:.2f}%")
    print(f"Speed      : {m['inference_ms']:.2f} ms/gambar")
    print('=' * 45)
else:
    print('Jalankan evaluasi dulu!')

## 🔥 STEP 8 — Visualisasi EigenCAM XAI

Menghasilkan heatmap yang menunjukkan **area mana yang dilihat model** saat mendeteksi huruf BISINDO.

In [ ]:
# Jalankan EigenCAM untuk semua gambar test set
BEST_PT = '/content/drive/MyDrive/bisindo_project/results/bisindo_yolo11n/weights/best.pt'
TEST_IMAGES = 'datasets/bisindo-dataset-1/test/images'
OUTPUT_DIR  = '/content/drive/MyDrive/bisindo_project/results/eigencam'

!python -m src.xai.eigencam \
    --weights {BEST_PT} \
    --source  {TEST_IMAGES} \
    --output  {OUTPUT_DIR} \
    --n-components 3

In [ ]:
# Tampilkan beberapa hasil EigenCAM secara langsung di Colab
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

eigencam_files = sorted(glob.glob(f'{OUTPUT_DIR}/*_eigencam_masked.png'))[:6]

if eigencam_files:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for i, path in enumerate(eigencam_files):
        img = mpimg.imread(path)
        axes[i].imshow(img)
        axes[i].set_title(path.split('/')[-1].replace('_eigencam_masked.png', ''), fontsize=10)
        axes[i].axis('off')
    plt.suptitle('Hasil EigenCAM XAI — Sampel 6 Gambar', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Belum ada hasil EigenCAM. Jalankan cell di atas dulu!')

## 🔍 BONUS — Coba Satu Gambar Spesifik

In [ ]:
# Ganti path ini ke gambar yang ingin dicoba
GAMBAR = 'datasets/bisindo-dataset-1/test/images/A_001.jpg'  # ← ganti sesuai nama file yang ada

import os
if not os.path.exists(GAMBAR):
    # Ambil gambar pertama yang ada di test set
    import glob
    files = glob.glob('datasets/bisindo-dataset-1/test/images/*.jpg')
    if files:
        GAMBAR = files[0]
        print(f'File A_001.jpg tidak ada, pakai: {GAMBAR}')
    else:
        print('❌ Tidak ada gambar di test set!')

BEST_PT = '/content/drive/MyDrive/bisindo_project/results/bisindo_yolo11n/weights/best.pt'

!python -m src.xai.eigencam \
    --weights {BEST_PT} \
    --source  {GAMBAR} \
    --output  /content/eigencam_preview

In [ ]:
# Tampilkan hasilnya
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg

hasil = glob.glob('/content/eigencam_preview/*.png')
if hasil:
    img = mpimg.imread(hasil[0])
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Belum ada hasil!')

---
## 📁 Struktur Hasil di Google Drive

Setelah semua step selesai, di Drive Anda akan ada:

```
MyDrive/bisindo_project/
├── results/
│   ├── bisindo_yolo11n/
│   │   ├── weights/
│   │   │   ├── best.pt     ← model terbaik (ini yang dipakai)
│   │   │   └── last.pt     ← checkpoint terakhir (untuk resume)
│   │   ├── results.png     ← grafik loss & mAP per epoch
│   │   └── confusion_matrix.png
│   ├── evaluation/
│   │   ├── metrics.json    ← angka-angka untuk skripsi
│   │   └── metrics.txt     ← ringkasan siap copy-paste
│   └── eigencam/
│       └── *.png           ← heatmap untuk setiap gambar test
```